# ministral_3_14b + qwen2_audio Multimodal Evaluation (eval)

Queries condition C SQL on combined DB. Writes xlsx to data/analysis_ministral_3_14b_qwen2_audio/.


In [1]:
import sys, os, sqlite3, json
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
for p in [ROOT] + list(ROOT.parents):
    if (p / '.gitignore').exists():
        ROOT = p; break
sys.path.insert(0, str(ROOT / 'backend/src'))
os.environ['PROJECT_ROOT'] = str(ROOT)
ABLATION_DIR = ROOT / 'data' / 'ablation_ministral_3_14b_qwen2_audio'
ANALYSIS_DIR = ROOT / 'data' / 'analysis_ministral_3_14b_qwen2_audio'
VIDEO_DIR = ROOT / 'data/videos/eval'
GT_PATH = ROOT / 'data/videos/eval/ground_truth.xlsx'
from service.impl.events_service_impl import queries_for_condition
from utils.database import setup_database
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Project root: {ROOT}')
print(f'Analysis dir: {ANALYSIS_DIR}')


Project root: /home/ghiffaryr/iseql/multimodal-surveillance-iseql
Analysis dir: /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/analysis_ministral_3_14b_qwen2_audio


In [2]:
# ── Check source DBs exist ──
audio_db = ROOT / 'data' / 'ablation_qwen2_audio' / f'qwen2_audio_w2_5s_h1_25s.db'
visual_db = ROOT / 'data' / 'ablation_ministral_3_14b' / f'ministral_3_14b.db'
missing = []
if not audio_db.exists():
    missing.append(str(audio_db))
if not visual_db.exists():
    missing.append(str(visual_db))
if missing:
    for m in missing:
        print(f'MISSING: {m}')
    raise FileNotFoundError(f'Source DBs missing: {missing}')
print(f'Audio DB: {audio_db}')
print(f'Visual DB: {visual_db}')


Audio DB: /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/ablation_qwen2_audio/qwen2_audio_w2_5s_h1_25s.db
Visual DB: /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/ablation_ministral_3_14b/ministral_3_14b.db


In [3]:
# ── Create combined DB ──
ablation_dir = ROOT / 'data' / 'ablation_ministral_3_14b_qwen2_audio'
ablation_dir.mkdir(parents=True, exist_ok=True)
combined_db = ablation_dir / 'ministral_3_14b_qwen2_audio_w2_5s_h1_25s.db'
if combined_db.exists():
    print(f'Combined DB exists: {combined_db}')
else:
    conn, cur = setup_database(combined_db)
    conn.execute("PRAGMA journal_mode=WAL")
    audio_conn = sqlite3.connect(str(audio_db))
    adf = pd.read_sql_query("SELECT * FROM SoundPerInterval", audio_conn)
    audio_conn.close()
    adf['AnalysisID'] = adf['AnalysisID'].apply(
        lambda x: f'ministral_3_14b_qwen2_audio_s{x.split("_")[-1][1:].split(".")[0]}' if 'ab' in x else x)
    adf.to_sql("SoundPerInterval", conn, if_exists="append", index=False)
    visual_conn = sqlite3.connect(str(visual_db))
    vdf = pd.read_sql_query("SELECT * FROM VisualPerInterval", visual_conn)
    participant_df = pd.read_sql_query("SELECT * FROM VisualParticipant", visual_conn)
    perframe_df = pd.read_sql_query("SELECT * FROM VisualPerFrame", visual_conn)
    rel_df = pd.read_sql_query("SELECT * FROM VisualRelation", visual_conn)
    visual_conn.close()
    vdf['AnalysisID'] = vdf['AnalysisID'].apply(
        lambda x: f'ministral_3_14b_qwen2_audio_s{x.split("_")[-1][1:].split(".")[0]}' if x.startswith('ministral_3_14b_') else x)
    perframe_df['AnalysisID'] = perframe_df['AnalysisID'].apply(
        lambda x: f'ministral_3_14b_qwen2_audio_s{x.split("_")[-1][1:].split(".")[0]}' if x.startswith('ministral_3_14b_') else x)
    rel_df['AnalysisID'] = rel_df['AnalysisID'].apply(
        lambda x: f'ministral_3_14b_qwen2_audio_s{x.split("_")[-1][1:].split(".")[0]}' if x.startswith('ministral_3_14b_') else x)
    conn.execute("DELETE FROM VisualPerInterval")
    vdf.to_sql("VisualPerInterval", conn, if_exists="append", index=False)
    participant_df.to_sql("VisualParticipant", conn, if_exists="append", index=False)
    perframe_df.to_sql("VisualPerFrame", conn, if_exists="append", index=False)
    rel_df.to_sql("VisualRelation", conn, if_exists="append", index=False)
    conn.commit()
    size_kb = os.path.getsize(combined_db) // 1024
    print(f'Combined DB created: {combined_db} ({size_kb} KiB)')
    conn.close()


Combined DB exists: /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/ablation_ministral_3_14b_qwen2_audio/ministral_3_14b_qwen2_audio_w2_5s_h1_25s.db


In [4]:
# ── Load GT ──
gt = pd.read_excel(GT_PATH, sheet_name='Ground Truth')
gt_visual = gt[gt['modality'] == 'visual'].dropna(subset=['scene'])
gt_visual['scene'] = gt_visual['scene'].astype(int)
gt_audio = gt[gt['modality'] == 'audio'].dropna(subset=['scene'])
gt_audio['scene'] = gt_audio['scene'].astype(int)
gt_audio['class'] = gt_audio['class'].apply(lambda c: c.strip().lower().replace(' ', '_') if isinstance(c, str) else c)
expected_df = pd.read_excel(GT_PATH, sheet_name='Expected Events')
print(f'GT: visual={len(gt_visual)} audio={len(gt_audio)} expected={len(expected_df)}')


GT: visual=47 audio=40 expected=30


In [5]:
# ── DELTAS for condition C ──
DELTAS = {
    "delta_visual_vehicle_escape": 50,
    "delta_visual_loitering": 150,
    "delta_visual_handoff": 240,
    "delta_sound_fight": 120,
    "delta_sound_vehicle_escape": 150,
    "delta_sound_vehicle_collision": 60,
}


In [6]:
# ── Multimodal event evaluation ──
conn = sqlite3.connect(str(combined_db))
event_rows = []
audio_evts = set(expected_df[expected_df['audio'].notna() & (expected_df['audio'] != '')]['event'].unique())
for _, row in expected_df.iterrows():
    aid = f'ministral_3_14b_qwen2_audio_s{row["scene"]}'
    evt = row['event']
    try:
        sql_map = queries_for_condition('C', DELTAS, analysis_id=aid)
        sql = sql_map.get(evt, 'SELECT 0 WHERE 1=0')
        df = pd.read_sql_query(sql, conn)
        det = not df.empty
        result = 'TP' if det else 'FN'
    except Exception as e:
        print(f'  {evt} query failed: {e}')
        det = False; result = 'ERROR'
    parts = []
    if evt in audio_evts:
        all_audio = conn.execute(
            'SELECT SoundClass, StartFrame, EndFrame FROM SoundPerInterval WHERE AnalysisID = ?',
            (aid,)
        ).fetchall()
        if all_audio:
            parts += [f'sound: {s}({sf}-{ef})' for s, sf, ef in all_audio]
    all_visual = conn.execute(
        'SELECT RelationType, StartFrame, EndFrame FROM VisualPerInterval WHERE AnalysisID = ?',
        (aid,)
    ).fetchall()
    if all_visual:
        parts += [f'visual: {r}({sf}-{ef})' for r, sf, ef in all_visual]
    if det and not parts:
        parts = [f'{evt} (query matched)']
    vis_rels = ', '.join(parts)
    event_rows.append({'scene': row['scene'], 'event': evt,
        'detected': 'YES' if det else 'NO', 'result': result,
        'relations': vis_rels})


# ── FP pass: check negative scenes (C) ──
all_scenes = sorted(expected_df['scene'].unique())
for evt in sorted(expected_df['event'].unique()):
    pos_scenes = set(expected_df[expected_df['event'] == evt]['scene'])
    for scene in all_scenes:
        if scene in pos_scenes:
            continue
        aid = f'ministral_3_14b_qwen2_audio_s{scene}'
        try:
            sql_map = queries_for_condition('C', DELTAS, analysis_id=aid)
            sql = sql_map.get(evt, 'SELECT 0 WHERE 1=0')
            df = pd.read_sql_query(sql, conn)
            if not df.empty:
                parts = []
                if evt in audio_evts:
                    all_audio = conn.execute('SELECT SoundClass, StartFrame, EndFrame FROM SoundPerInterval WHERE AnalysisID = ?', (aid,)).fetchall()
                    if all_audio:
                        parts += [f'sound: {s}({sf}-{ef})' for s, sf, ef in all_audio]
                all_visual = conn.execute('SELECT RelationType, StartFrame, EndFrame FROM VisualPerInterval WHERE AnalysisID = ?', (aid,)).fetchall()
                if all_visual:
                    parts += [f'visual: {r}({sf}-{ef})' for r, sf, ef in all_visual]
                rel_str = ', '.join(parts) if parts else '|'
                event_rows.append({'scene': scene, 'event': evt, 'detected': 'YES', 'result': 'FP', 'relations': rel_str})
        except:
            pass

conn.close()
edf = pd.DataFrame(event_rows)
edf['relations'] = edf['relations'].fillna('')
# Summary metrics
metrics = []
for evt in sorted(expected_df['event'].unique()):
    sub = edf[edf['event'] == evt]
    tpp = len(sub[sub['result'] == 'TP'])
    fpp = len(sub[sub['result'] == 'FP'])
    fnn = len(sub[sub['result'] == 'FN'])
    support = tpp + fnn
    p = tpp / (tpp + fpp) if (tpp + fpp) > 0 else 0.0
    r = tpp / (tpp + fnn) if (tpp + fnn) > 0 else 0.0
    f1 = 2*p*r/(p+r) if (p+r) > 0 else 0.0
    metrics.append({'event': evt, 'precision': round(p, 3), 'recall': round(r, 3), 'f1': round(f1, 3), 'TP': tpp, 'FP': fpp, 'FN': fnn, 'support': support})
metrics_df = pd.DataFrame(metrics)
print('\n=== Multimodal Event Summary ===')
print(metrics_df.to_string(index=False))
xlsx_path = ANALYSIS_DIR / 'multimodal_event_eval_ministral_3_14b_qwen2_audio_w2_5s_h1_25s.xlsx'
with pd.ExcelWriter(xlsx_path) as writer:
    metrics_df.to_excel(writer, sheet_name='Summary', index=False)
    for sc in sorted(expected_df['scene'].unique()):
        sc_df = edf[edf['scene'] == sc]
        if not sc_df.empty:
            sc_df.to_excel(writer, sheet_name=f'Scene_{sc}', index=False)
print(f'Written: {xlsx_path}')
# Ablation summary
tp = len(edf[edf['result'] == 'TP'])
fp = len(edf[edf['result'] == 'FP'])
fn = len(edf[edf['result'] == 'FN'])
support = tp + fn
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
result_df = pd.DataFrame([{'combo': 'ministral_3_14b_qwen2_audio',
    'audio': 'qwen2_audio_w2.5s_h1.25s',
    'visual': 'ministral_3_14b',
    'precision': round(precision, 3), 'recall': round(recall, 3), 'f1': round(f1, 3), 'TP': tp, 'FP': fp, 'FN': fn, 'support': support}])
result_df.to_excel(ANALYSIS_DIR / 'summary.xlsx', index=False)
print(f'Summary: P={precision:.3f} R={recall:.3f} F1={f1:.3f} TP={tp} FP={fp} FN={fn}')



=== Multimodal Event Summary ===
               event  precision  recall    f1  TP  FP  FN  support
               fight      1.000     0.8 0.889   4   0   1        5
gunshot_or_explosion      1.000     1.0 1.000   5   0   0        5
             handoff      1.000     1.0 1.000   5   0   0        5
           loitering      0.714     1.0 0.833   5   2   0        5
   vehicle_collision      0.714     1.0 0.833   5   2   0        5
      vehicle_escape      0.714     1.0 0.833   5   2   0        5


Written: /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/analysis_ministral_3_14b_qwen2_audio/multimodal_event_eval_ministral_3_14b_qwen2_audio_w2_5s_h1_25s.xlsx
Summary: P=0.829 R=0.967 F1=0.892 TP=29 FP=6 FN=1
